# 02 · Table Geometry ML — does the floor plan sell wine?
Learns how **distance to kitchen/bar/pool, seats, outdoor** move avg check and wine attach — the notebook version of `/analytics/table-performance` driver weights (ridge). Here we add nonlinear ML + permutation importance + partial dependence.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from wineops_data import get_checks, get_tables, get_consumption, get_orders, get_inventory, daily_series
plt.rcParams['figure.figsize'] = (11, 4)

checks, tables = get_checks(), get_tables()
checks['items'] = checks['items'].apply(lambda x: x if isinstance(x, list) else [])
checks['has_wine'] = checks['items'].apply(lambda it: any(i.get('is_wine') for i in it))
per_table = checks.groupby('table_id').agg(avg_check=('total','mean'), checks=('total','size'), attach=('has_wine','mean')).reset_index()
df = per_table.merge(tables, left_on='table_id', right_on='id')
FEATS = ['distance_to_kitchen_m','distance_to_bar_m','distance_to_pool_m','seats','is_outdoor']
df['is_outdoor'] = df['is_outdoor'].astype(int)
df[FEATS + ['avg_check','attach']].describe().round(2)

In [ ]:
# Linear view first (mirrors the production ridge): standardized coefficients
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
X, yv = StandardScaler().fit_transform(df[FEATS]), df['avg_check']
ridge = Ridge(alpha=1.0).fit(X, yv)
pd.Series(ridge.coef_, index=FEATS).sort_values().plot.barh(title='Standardized effect on avg check ($)')
plt.show()

In [ ]:
# Nonlinear: random forest + permutation importance (honest importance)
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
rf = RandomForestRegressor(n_estimators=400, random_state=0).fit(df[FEATS], yv)
imp = permutation_importance(rf, df[FEATS], yv, n_repeats=30, random_state=0)
pd.Series(imp.importances_mean, index=FEATS).sort_values().plot.barh(title='Permutation importance (avg check)')
plt.show()

In [ ]:
# Partial dependence: HOW the strongest distance drives spend
from sklearn.inspection import PartialDependenceDisplay
PartialDependenceDisplay.from_estimator(rf, df[FEATS], ['distance_to_bar_m','distance_to_kitchen_m'])
plt.show()

In [ ]:
# Same machinery for wine ATTACH RATE (the wine program's lever)
rf2 = RandomForestRegressor(n_estimators=400, random_state=0).fit(df[FEATS], df['attach'])
imp2 = permutation_importance(rf2, df[FEATS], df['attach'], n_repeats=30, random_state=0)
pd.Series(imp2.importances_mean, index=FEATS).sort_values().plot.barh(title='Permutation importance (wine attach rate)')
plt.show()

**Read:** a negative bar-distance effect with meaningful importance = seat high-value parties near the bar and route the wine cart there first. With live data, add weekday × zone interactions and re-check — synthetic data plants a bar-proximity effect on purpose so you can validate the pipeline.